In [1]:
import pandas as pd

from collections import defaultdict

import json
import os

In [2]:
texts = defaultdict(list)

with open("../../dataset/final_dataset/jobs.json", 'r') as f:
    data = json.load(f)

for item in data:
    texts["id"].append(item.get("humanjobid", ""))
    texts["company"].append(item.get("companytext", ""))
    texts["job title"].append(item.get("jobtitle", ""))
    texts["text"].append(item.get("searchtext", ""))

main_df = pd.DataFrame(texts)
main_df.head()

,id,company,job title,text
0,1527392,Jobindex,"IT-administrator – få indflydelse på et setup,...",Vil du ind i en virksomhed i rivende udvikling...
1,1527395,Jobindex,"IT-administrator – få indflydelse på et setup,...","IT-administrator – få indflydelse på et setup,..."
2,1527397,Aqua d'Or Mineral Water A/S,SQE Manager,For jobsøgere For arbejdsgivere Aqua d'Or Mi...
3,1527417,Klimabrands,Kundeservice / teknisk support,For jobsøgere For arbejdsgivere mailto:job@k...
4,1527440,Scan Studio ApS,Retail designer med teknikken på plads,For jobsøgere For arbejdsgivere Scan Studio ...


In [9]:
import os
import json
import pandas as pd
import glob

# Assuming main_df is your master dataframe
todo = {m: {p: [] for p in ["structured", "semi-structured", "unstructured"]} for m in ["qwen", "gemma", "llama"]}

# Pre-convert ID to string for robust matching across all folders
main_df['id_str'] = main_df['id'].astype(str)

for model in ["qwen", "gemma", "llama"]:
    for prompt in ["structured", "semi-structured", "unstructured"]:
        
        col_name = f"triples_{model}_{prompt}"
        if col_name not in main_df.columns:
            main_df[col_name] = None
            
        # Process Range-based Chunks from ./logs (Strict Index Alignment)
        series_chunks = []
        ranges = [(0, 3000), (3000, 6000), (6000, 8500), (7500, 10585)]
        
        for r in ranges:
            filepath = os.path.join("./logs", f"temporary_results_{model}_{prompt}_{r[0]}_{r[1]}.txt")
            if os.path.exists(filepath):
                try:
                    with open(filepath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if data:
                        df_c = pd.DataFrame(data)
                        if 'triples' in df_c.columns:
                            s = df_c['triples'].copy()
                            s.index = range(r[0], r[0] + len(s))
                            series_chunks.append(s)
                except Exception: pass

        if series_chunks:
            combined_range = pd.concat(series_chunks)
            combined_range = combined_range[~combined_range.index.duplicated(keep='last')]
            valid_idx = combined_range.index.intersection(main_df.index)
            main_df.loc[valid_idx, col_name] = combined_range.loc[valid_idx].values

        # Process ID-based Batches from ../logs_2 and ../logs_3
        # We look for ANY file starting with the model/prompt pattern
        for folder in ["../logs_2", "../logs_3", "../logs_4", "../logs_5", "../logs_6", "../logs_7"]:
            # Pattern matches: temporary_results_model_prompt_*.txt
            search_pattern = os.path.join(folder, f"temporary_results_{model}_{prompt}_*.txt")
            batch_files = glob.glob(search_pattern)
            
            # If no range-suffix files found, check for the base filename too
            base_file = os.path.join(folder, f"temporary_results_{model}_{prompt}.txt")
            if os.path.exists(base_file):
                batch_files.append(base_file)

            for batch_path in batch_files:
                if os.path.getsize(batch_path) == 0:
                    continue
                    
                try:
                    with open(batch_path, "r", encoding="utf-8") as f:
                        data_batch = json.load(f)
                    if data_batch:
                        df_batch = pd.DataFrame(data_batch)
                        
                        if 'id' in df_batch.columns and 'triples' in df_batch.columns:
                            df_batch = df_batch.drop_duplicates(subset=['id'], keep='last')
                            
                            # Create mapping (ID string -> Triple)
                            id_map = dict(zip(df_batch['id'].astype(str), df_batch['triples']))
                            
                            # Update main_df: fill existing NaNs with these batch values
                            new_data = main_df['id_str'].map(id_map)
                            main_df[col_name] = main_df[col_name].fillna(new_data)
                except Exception as e:
                    print(f"Error in {batch_path}: {e}")

        # Use fillna(None) to ensure we can identify missing rows reliably
        todo[model][prompt] = main_df[main_df[col_name].isna()].index.tolist()

# Cleanup
main_df.drop(columns=['id_str'], inplace=True)

with open("./todo.json", "w+", encoding="utf-8") as f:
    json.dump(todo, f, indent=4)

In [10]:
import os
import json
import glob
import pandas as pd

def check_log_overlaps(folder_a="../logs_2", folder_b="../logs_3"):
    models = ["qwen", "gemma", "llama"]
    prompts = ["structured", "semi-structured", "unstructured"]
    
    overlap_report = []

    for model in models:
        for prompt in prompts:
            # Gather IDs from Folder A
            ids_a = set()
            files_a = glob.glob(os.path.join(folder_a, f"temporary_results_{model}_{prompt}_*.txt"))
            for f in files_a:
                if os.path.getsize(f) > 0:
                    try:
                        with open(f, 'r', encoding='utf-8') as src:
                            data = json.load(src)
                            ids_a.update([str(item['id']) for item in data if 'id' in item])
                    except: pass

            # Gather IDs from Folder B
            ids_b = set()
            files_b = glob.glob(os.path.join(folder_b, f"temporary_results_{model}_{prompt}_*.txt"))
            for f in files_b:
                if os.path.getsize(f) > 0:
                    try:
                        with open(f, 'r', encoding='utf-8') as src:
                            data = json.load(src)
                            ids_b.update([str(item['id']) for item in data if 'id' in item])
                    except: pass

            # Calculate Intersection
            common_ids = ids_a.intersection(ids_b)
            
            if common_ids or ids_a or ids_b:
                overlap_report.append({
                    "model": model,
                    "prompt": prompt,
                    "count_logs_2": len(ids_a),
                    "count_logs_3": len(ids_b),
                    "overlap_count": len(common_ids),
                    "overlap_percentage": round((len(common_ids) / len(ids_a) * 100), 2) if ids_a else 0
                })

    return pd.DataFrame(overlap_report)

# Execute and view the report
df_overlap = check_log_overlaps()
print(df_overlap)

Empty DataFrame
Columns: []
Index: []


In [11]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10584 entries, 0 to 10583
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             10584 non-null  int64 
 1   company                        10584 non-null  object
 2   job title                      10584 non-null  object
 3   text                           10582 non-null  object
 4   triples_qwen_structured        10584 non-null  object
 5   triples_qwen_semi-structured   10584 non-null  object
 6   triples_qwen_unstructured      10584 non-null  object
 7   triples_gemma_structured       10584 non-null  object
 8   triples_gemma_semi-structured  10584 non-null  object
 9   triples_gemma_unstructured     10584 non-null  object
 10  triples_llama_structured       10584 non-null  object
 11  triples_llama_semi-structured  10584 non-null  object
 12  triples_llama_unstructured     10584 non-null  object
dtypes

In [15]:
main_df.to_excel("LLM_outputs.xlsx")

In [12]:
for k in todo:
    print(k)
    for p, v in todo[k].items():
        print(p, len(v))

qwen
structured 0
semi-structured 0
unstructured 0
gemma
structured 0
semi-structured 0
unstructured 0
llama
structured 0
semi-structured 0
unstructured 0


In [14]:
todo

{'qwen': {'structured': [], 'semi-structured': [], 'unstructured': []},
 'gemma': {'structured': [], 'semi-structured': [], 'unstructured': []},
 'llama': {'structured': [], 'semi-structured': [], 'unstructured': []}}